In [ ]:
import optuna
import pandas as pd
import matplotlib.pyplot as plt

from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.feature_selection import mutual_info_classif

# Data Path
target_name = 'Survived'
train_url = '/kaggle/input/competitions/titanic/train.csv'
test_url = '/kaggle/input/competitions/titanic/test.csv'

# Read the data
X_full = pd.read_csv(train_url)
X_test_full = pd.read_csv(test_url)

X_full.head(20)

In [ ]:
# Remove rows with missing target, separate target from predictors
X_full.dropna(axis=0, subset=[target_name], inplace=True)
y = X_full[target_name]
X_full.drop([target_name], axis=1, inplace=True)

X_full.head(20)

In [ ]:

missing_val_count_by_column = X_full.isnull().sum()
missing_cols = missing_val_count_by_column[missing_val_count_by_column > 0]

missing_df = pd.DataFrame({
    'Missing Values': missing_cols,
    'Percentage (%)': (missing_cols / len(X_full)) * 100
}).sort_values(by='Missing Values', ascending=False)

print("Missing values table for training data")
print("-"*40)
print(missing_df)


missing_val_count_by_column = X_test_full.isnull().sum()
missing_cols = missing_val_count_by_column[missing_val_count_by_column > 0]

print("="*40)

missing_df = pd.DataFrame({
    'Missing Values': missing_cols,
    'Percentage (%)': (missing_cols / len(X_test_full)) * 100
}).sort_values(by='Missing Values', ascending=False)

print("Missing values table for test data")
print("-"*40)
print(missing_df)

In [ ]:
for df in [X_full, X_test_full]:
    df['Title'] = df['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
    df['Title'] = df['Title'].replace(['Lady', 'Countess', 'Capt', 'Col', 'Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona'], 'Rare')
    df['Title'] = df['Title'].replace({'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs'})

    df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
    df['IsAlone'] = (df['FamilySize'] == 1).astype(int)

    # df['Deck'] = df['Cabin'].str.slice(0, 1).fillna('U')


title_age_medians = X_full.groupby('Title')['Age'].median()

for title, median_age in title_age_medians.items():
    X_full.loc[(X_full['Age'].isnull()) & (X_full['Title'] == title), 'Age'] = median_age
    X_test_full.loc[(X_test_full['Age'].isnull()) & (X_test_full['Title'] == title), 'Age'] = median_age

X_full['Age'] = X_full['Age'].fillna(X_full['Age'].median())
X_test_full['Age'] = X_test_full['Age'].fillna(X_full['Age'].median())

median_fare_p3 = X_full[X_full['Pclass'] == 3]['Fare'].median()
X_test_full['Fare'] = X_test_full['Fare'].fillna(median_fare_p3)

# for df in [X_full, X_test_full]:
#     df['Is_Child'] = (df['Age'] < 10).astype(int)
#     df['Is_Woman_Or_Child'] = ((df['Sex'] == 'female') | (df['Is_Child'] == 1)).astype(int)

X_full.head(20)

In [ ]:
# Select numerical columns
numerical_cols = [cname for cname in X_full.columns if X_full[cname].dtype in ['int64', 'float64']]

# Select categorical columns with relatively low cardinality (convenient but arbitrary)
categorical_cols = [cname for cname in X_full.columns if X_full[cname].nunique() < 10 and X_full[cname].dtype == "object"]

# Select columns with empty cells
cols_with_missing = [col for col in X_full.columns if X_full[col].isnull().any()]

if 'PassengerId' in numerical_cols:
  numerical_cols.remove('PassengerId')

# Keep selected columns only
my_cols = categorical_cols + numerical_cols
X = X_full[my_cols].copy()
X_test = X_test_full[my_cols].copy()

X.head(10)

In [ ]:
# Preprocessing for numerical data
numerical_transformer = SimpleImputer(strategy='mean', add_indicator=True)

# Preprocessing for categorical data
categorical_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='most_frequent', add_indicator=True)),
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ]
)

# Bundle preprocessing for numerical and categorical data
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_cols),
        ('cat', categorical_transformer, categorical_cols)
    ]
)

print("Preprocessing completed successfully")

In [ ]:
# 5. Calculating and graphing Mutual Information (MI)
print('--- Calculating & Plotting Mutual Information ---')

X_transformed = preprocessor.fit_transform(X)
feature_names = preprocessor.get_feature_names_out()

mi_scores = mutual_info_classif(X_transformed, y, random_state=0)
mi_series = pd.Series(mi_scores, index=feature_names).sort_values(ascending=True)

plt.figure(figsize=(10, round(mi_series.count() * 0.3)), dpi=100)
mi_series.plot(kind='barh', color='skyblue', edgecolor='black')
plt.title('Mutual Information Scores', fontsize=12, pad=15)
plt.xlabel('MI Score', fontsize=10)
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

print("Top Feature Scores:")
print(mi_series.sort_values(ascending=False))

In [ ]:
def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 500, step=25),
        'max_depth': trial.suggest_int('max_depth', 2, 4),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.08, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 0.85),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 0.85),
        'gamma': trial.suggest_float('gamma', 0.1, 2.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 1.0, 10.0),
        'random_state': 0,
        'eval_metric': 'logloss'
    }

    model = XGBClassifier(**params)

    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', model)
    ])

    score = cross_val_score(pipeline, X, y, cv=5, scoring='accuracy', n_jobs=-1).mean()
    
    return score

In [ ]:
optuna.logging.set_verbosity(optuna.logging.WARNING)

study = optuna.create_study(direction='maximize')

print("The search is on for the best parameter combination...")

study.optimize(objective, n_trials=50)

print("\n" + "="*40)
print(f"🔥 Best CV Accuracy: {study.best_value:.4f}")
print("🎯 Best Parameters:")
for key, value in study.best_params.items():
    print(f"    {key}: {value}")

print("="*40)

In [ ]:
best_model = XGBClassifier(**study.best_params, random_state=0)

# Bundle preprocessing and modeling code in a pipeline
final_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', best_model)
])

final_pipeline.fit(X, y)

test_preds = final_pipeline.predict(X_test)

print("Model fit completed successfully")

In [ ]:
output = pd.DataFrame({'PassengerId': X_test_full.PassengerId, target_name: test_preds})

output.to_csv('submission.csv', index=False)

print("Your submission was successfully saved!")

In [ ]:
# import pandas as pd
# from sklearn.model_selection import train_test_split
# from sklearn.compose import ColumnTransformer
# from sklearn.pipeline import Pipeline
# from sklearn.impute import SimpleImputer
# from sklearn.preprocessing import OneHotEncoder
# from sklearn.ensemble import RandomForestClassifier
# from sklearn.metrics import accuracy_score
# from xgboost import XGBClassifier
# from sklearn.feature_selection import mutual_info_classif
# import matplotlib.pyplot as plt

# target_name = 'Survived'
# train_url = '/kaggle/input/competitions/titanic/train.csv'
# test_url = '/kaggle/input/competitions/titanic/test.csv'

# X_full = pd.read_csv(train_url)
# X_test_full = pd.read_csv(test_url)

# X_full.head(10)

In [ ]:
# X_full.dropna(axis=0, subset=[target_name], inplace=True)
# y = X_full[target_name]
# X_full.drop([target_name], axis=1, inplace=True)

# X_train_full, X_valid_full, y_train, y_valid = train_test_split(X_full, y, train_size=0.8, test_size=0.2,random_state=0)

# numerical_cols = [cname for cname in X_train_full.columns if X_train_full[cname].dtype in ['int64', 'float64']]
# categorical_cols = [cname for cname in X_train_full.columns if X_train_full[cname].nunique() < 10 and X_train_full[cname].dtype == "object"]

# if 'PassengerId' in numerical_cols:
#     numerical_cols.remove('PassengerId')

# my_cols = categorical_cols + numerical_cols
# X_train = X_train_full[my_cols].copy()
# X_valid = X_valid_full[my_cols].copy()
# X_test = X_test_full[my_cols].copy()

# X_train.head()

In [ ]:
# numerical_transformer = SimpleImputer(strategy='mean', add_indicator=True)

# categorical_transformer = Pipeline(
#     steps=[
#         ('imputer', SimpleImputer(strategy='most_frequent', add_indicator=True)),
#         ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
#     ]
# )

# preprocessor = ColumnTransformer(
#     transformers=[
#         ('num', numerical_transformer, numerical_cols),
#         ('cat', categorical_transformer, categorical_cols)
#     ]
# )

# print("Preprocessing completed successfully")

In [ ]:
# print('--- Calculating & Plotting Mutual Information ---')

# X_train_transformed = preprocessor.fit_transform(X_train)
# feature_names = preprocessor.get_feature_names_out()

# mi_scores = mutual_info_classif(X_train_transformed, y_train, random_state=0)
# mi_series = pd.Series(mi_scores, index=feature_names).sort_values(ascending=True)

# plt.figure(figsize=(10, round(mi_series.count() * 0.3)), dpi=100)
# mi_series.plot(kind='barh', color='skyblue', edgecolor='black')
# plt.title('Mutual Information Scores', fontsize=12, pad=15)
# plt.xlabel('MI Score', fontsize=10)
# plt.grid(axis='x', linestyle='--', alpha=0.7)
# plt.tight_layout()
# plt.show()

# print("Top Feature Scores:")
# print(mi_series.sort_values(ascending=False))

In [ ]:
# model = XGBClassifier(
#     n_estimators=500,
#     max_depth=3,
#     random_state=0,
#     learning_rate=0.01,
#     subsample=0.8,
#     colsample_bytree=0.8,
#     gamma=0.1,
#     reg_lambda=1.5
# )

# my_pipeline = Pipeline(
#     steps=[
#         ('preprocessor', preprocessor),
#         ('model', model)
#     ]
# )

# my_pipeline.fit(X_train, y_train)
# preds = my_pipeline.predict(X_valid)
# print('Accuracy:', accuracy_score(y_valid, preds))

In [ ]:
# my_pipeline.fit(X_full, y)
# # preds_test = my_pipeline.predict(X_test)

# output = pd.DataFrame({'PassengerId': X_test_full.PassengerId, target_name: preds_test})
# output.to_csv('submission.csv', index=False)
# print("Your submission was successfully saved!")